In [11]:
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.svm import SVR
from sklearn.ensemble import (ExtraTreesRegressor, HistGradientBoostingRegressor,
                              VotingRegressor)
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [ ]:
COLS = ["Time Frame", "Date", "Ta (C)", "Tm (C)", "I3 (W/m2)", "I15 (W/m2)", "P (kW)"]

def add_features(df):
    dt = pd.to_datetime(df["Date"], format="%m/%d/%y %I:%M %p")
    tf = df["Time Frame"].values
    doy = dt.dt.dayofyear.values
    X = pd.DataFrame({
        "Ta": df["Ta (C)"], "Tm": df["Tm (C)"],
        "I3": df["I3 (W/m2)"], "I15": df["I15 (W/m2)"],
        "h_sin": np.sin(2*np.pi*tf/24), "h_cos": np.cos(2*np.pi*tf/24),
        "d_sin": np.sin(2*np.pi*doy/365.25), "d_cos": np.cos(2*np.pi*doy/365.25),
    })
    return X.astype(np.float64)

def load_train(path):
    df = pd.read_csv(path)
    dt = pd.to_datetime(df["Date"], format="%m/%d/%y %I:%M %p")

    # righe anomale: timestamp congelato/duplicato, sensori a 0, ora incoerente
    bad = (dt.duplicated(keep="first")
           | ((df["Ta (C)"] == 0) & (df["Tm (C)"] == 0))
           | ((df["Time Frame"] % 24) != dt.dt.hour))
    ok = ~bad.values

    tf = df["Time Frame"].values
    # coppia (i -> i+1) valida solo se entrambe le righe sono buone
    # e se i+1 è davvero l'ora successiva
    valid = ok[:-1] & ok[1:] & (tf[1:] == tf[:-1] % 24 + 1)

    X = add_features(df).iloc[:-1][valid].reset_index(drop=True)
    y = df["P (kW)"].values[1:][valid].astype(np.float64)
    return X, y

X, y = load_train("./DBs/SolarPark/train.txt")
print(X.shape, "std(y) =", y.std())

(5443, 8) std(y) = 249.66906920371883


In [10]:
def wrap(m): # standardizza sia i pattern che le label
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), m),
        transformer=StandardScaler()
    )

models = {
    "svr": wrap(SVR(C=10, epsilon=0.05, gamma="scale")),
    "extra": ExtraTreesRegressor(500, min_samples_leaf=2, n_jobs=-1, random_state=0),
    "hgb": HistGradientBoostingRegressor(learning_rate=0.05, max_iter=500,
                                         max_leaf_nodes=15, l2_regularization=1.0,
                                         random_state=0),
}

# scegli in base al test: TimeSeriesSplit se il test è "nel futuro",
# altrimenti KFold a blocchi contigui (shuffle=False)
cv = TimeSeriesSplit(5) # KFold(n_splits=5, shuffle=False)

for name, m in models.items():
    s = cross_val_score(m, X, y, cv=cv, scoring="neg_mean_squared_error", n_jobs=-1)
    print(f"{name:6s} RMSE = {np.sqrt(-s).mean():.2f} ± {np.sqrt(-s).std():.2f}")

svr    RMSE = 159.49 ± 67.68
extra  RMSE = 77.90 ± 11.63
hgb    RMSE = 82.02 ± 13.79


In [12]:
svr = wrap(SVR())
svr_grid = {
    "regressor__svr__C":       [1, 3,4, 5,6,10],
    "regressor__svr__epsilon": [0.01, 0.05,0.06,0.04, 0.055, 0.045],
    "regressor__svr__gamma":   [0.02, 0.03, 0.035,0.04 , 0.025]
}
svr_gs = GridSearchCV(svr, svr_grid, cv=cv, scoring="neg_mean_squared_error", n_jobs=-1)
svr_gs.fit(X, y)
print(np.sqrt(-svr_gs.best_score_), svr_gs.best_params_)

rf = RandomForestRegressor(random_state=0, n_jobs=1)
rf_grid = {
    "n_estimators":     [300, 600],
    "max_depth":        [None, 12, 20],
    "min_samples_leaf": [1, 2, 4],
    "max_features":     [0.5, 0.8, 1.0],
}
rf_gs = GridSearchCV(rf, rf_grid, cv=cv, scoring="neg_mean_squared_error", n_jobs=-1)
rf_gs.fit(X, y)
print("RF:", np.sqrt(-rf_gs.best_score_), rf_gs.best_params_)

xgb = XGBRegressor(objective="reg:squarederror", tree_method="hist",
                   random_state=0, n_jobs=1)
xgb_grid = {
    "n_estimators":     [300, 600],
    "learning_rate":    [0.03, 0.05, 0.1],
    "max_depth":        [3, 5, 7],
    "min_child_weight": [1, 5],
    "subsample":        [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0],
}
xgb_gs = GridSearchCV(xgb, xgb_grid, cv=cv, scoring="neg_mean_squared_error", n_jobs=-1)
xgb_gs.fit(X, y)
print("XGB:", np.sqrt(-xgb_gs.best_score_), xgb_gs.best_params_)

133.9396046360225 {'regressor__svr__C': 1, 'regressor__svr__epsilon': 0.06, 'regressor__svr__gamma': 0.02}
RF: 77.68036946577962 {'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 4, 'n_estimators': 600}
XGB: 78.8068608589322 {'colsample_bytree': 0.7, 'learning_rate': 0.03, 'max_depth': 3, 'min_child_weight': 5, 'n_estimators': 300, 'subsample': 0.7}


In [13]:
ens = VotingRegressor([
    ("svr", svr_gs.best_estimator_),
    ("random", rf_gs.best_estimator_),
    ("xgb", xgb_gs.best_estimator_),
])
s = cross_val_score(ens, X, y, cv=cv, scoring="neg_mean_squared_error", n_jobs=-1)
print("ensemble RMSE:", np.sqrt(-s).mean())

ens.fit(X, y)
test = pd.read_csv("./DBs/SolarPark/test.txt")
X_test = add_features(test)           # NON scartare righe del test
pred = np.clip(ens.predict(X_test), 0, None)   # la potenza non è negativa

ensemble RMSE: 83.22147159863451
